<a href="https://colab.research.google.com/github/djduncanlaw/AISV-819-LLM-Fundamentals-and-Practical-Applications/blob/main/C1_McK_RAG_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Hi and hello")

Hi and hello


# Class 1: Simple RAG Example with LangChain

This notebook demonstrates the basic flow of Retrieval-Augmented Generation (RAG) using LangChain:
1. **Load**: Download and parse a PDF document
2. **Split**: Chunk the document into smaller pieces
3. **Embed & Store**: Create embeddings and store in a vector database
4. **Query**: Retrieve relevant chunks and generate responses

**What you'll learn:**
- Build a basic RAG pipeline from scratch using LangChain
- Load, split, embed, and store PDF documents in a vector database (LanceDB)
- Create a retrieval-augmented generation chain using LCEL
- Query the system and get context-grounded answers

**Prerequisites:** `pip install -r requirements.txt` and set `OPENAI_API_KEY` in your environment.

In [ ]:
%%writefile requirements.txt
langchain-core
langchain-openai
langchain-community
langchain-text-splitters
lancedb
pypdf

Writing requirements.txt


In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.2/75.2 MB 65.8 MB/s  0:00:01
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import LanceDB
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

/tmp/ipykernel_3140/1671462340.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Step 1: Load the PDF Document

We'll use LangChain's PyPDFLoader to download and parse the PDF file.

In [ ]:
# Load the PDF from URL
pdf_file = "McK_Apr2026_the-ai-transformation-manifesto.pdf"
loader = PyPDFLoader(pdf_file)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF")

Loaded 6 pages from the PDF


## Step 2: Split into Chunks

Long documents need to be split into smaller chunks for effective retrieval. We use `RecursiveCharacterTextSplitter` which tries to split on natural boundaries (paragraphs, sentences) while respecting the chunk size.

Here we use chunk size of 1000 characters with a 200 characters overlap.

In [ ]:
# Split documents into chunks, use Python's len function for character count
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=500,
    length_function=len,
)

chunks = text_splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")

Split into 24 chunks


> **Note:** A chunk size of 1,000 characters with 200-character overlap is a common starting point for text splitting. The overlap ensures that sentences spanning a chunk boundary aren't lost. Smaller chunks improve retrieval precision but may lack context; larger chunks retain more context but may dilute relevance signals. Tune these values based on your document structure and query patterns.

## Step 3: Create Embeddings and Vector Store

We embed all chunks using OpenAI's embedding model and store them in LanceDB, a serverless vector database with persistent storage.

In [ ]:
from google.colab import userdata
oak = userdata.get('OPENAI_API_KEY')

In [ ]:
# Create embeddings and vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=oak)
vectorstore = LanceDB.from_documents(chunks, embeddings)

print(f"Created vector store with {len(chunks)} documents")

Created vector store with 24 documents


## Step 4: RAG Query - Retrieval + Generation

Using LCEL (LangChain Expression Language), we create a chain that:
1. Takes a question
2. Retrieves relevant documents
3. Formats them as context
4. Passes to the LLM for answering

In [ ]:
# Create the RAG chain using LCEL (LangChain Expression Language)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=oak)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:

{context}

Question: {question}"""
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# Ask questions from the article
questions = [
    "List 12 ideas companies need to follow from Mckinsey's AI transformation manifesto? List them using numbers",
    "Summarize McKinsey's AI transformation manifesto in 3 sentences."
]

for q in questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)

Q: List 12 ideas companies need to follow from Mckinsey's AI transformation manifesto? List them using numbers
A: 1. (Re)learn like your business depends on it.
2. Design for adoption and build for scale.
3. If the value you’re creating doesn’t move the business, you’re getting it wrong.
4. Building the tech and AI muscle of your senior business leaders should be a top priority.
5. Economic leverage points are your best focal points.
6. Make data easy to consume—and enrich it for advantage.
7. Focus on one to three business domains for AI reinvention.
8. Ensure clear accountability for business KPIs that matter most.
9. Invest substantially and stage-gate your investments.
10. Foster a maniacal focus on customers/users.
11. Encourage creative problem-solving and coordinated use of tech and nontech levers.
12. Drive the tech agenda from the business leadership level.
------------------------------------------------------------
Q: Summarize McKinsey's AI transformation manifesto in 3 sen

In [ ]:
#@title Query Box { run: "auto" }

# 1. This creates the text input box
your_query = "Summarize McKinsey's AI transformation manifesto in 1 sentence" #@param {type:"string"}

def get_response(your_query):
  answer = rag_chain.invoke(your_query)
  return answer

# 3. Display the response
response = get_response(your_query)
print("Response:", response)


Response: McKinsey's AI transformation manifesto emphasizes that organizations must rapidly learn, adopt, and scale AI technologies while focusing on strategic business goals, data quality, and leadership involvement to achieve significant competitive advantages and transformative results.


# Simple UI with widgets

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Create the UI elements
query_input = widgets.Text(
    value='',
    placeholder='Type your query here...',
    description='Query:',
    layout=widgets.Layout(width='70%')
)

submit_button = widgets.Button(
    description='Submit',
    button_style='success',
    tooltip='Click to get response'
)

clear_button = widgets.Button(
    description='Clear',
    button_style='warning',
    tooltip='Click to clear input and response'
)

# A styled Output widget with borders and padding
response_output = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ced4da',
        padding='15px',
        margin='10px 0 0 0',
        border_radius='5px',
        background_color='#1e1e1e', # Changed to a dark background for dark mode
        min_height='100px'
    )
)

# 2. Define what happens when you click submit
def on_submit_clicked(b):
    with response_output:
        clear_output() # Clears the previous response
        query = query_input.value

        if query.strip() == "":
            display(widgets.HTML(value="<b style='color: #ff6b6b;'>Please enter a valid query.</b>"))
        else:
            # Display a beautiful loading spinner while waiting for response
            spinner_html = """
            <div style='display: flex; align-items: center; font-family: Arial, sans-serif;'>
                <div style='
                    border: 4px solid #f3f3f3;
                    border-top: 4px solid #2ca02c;
                    border-radius: 50%;
                    width: 24px;
                    height: 24px;
                    animation: spin 1s linear infinite;
                    margin-right: 12px;
                '></div>
                <span style='color: #ffffff;'>Generating answer... Please wait</span>
                <style>
                    @keyframes spin {
                        0% { transform: rotate(0deg); }
                        100% { transform: rotate(360deg); }
                    }
                </style>
            </div>
            """
            display(widgets.HTML(value=spinner_html))

            # Fetch response
            response_text = get_response(query)

            # Clear the loading spinner and display the actual response
            clear_output()
            html_content = f"""
            <div style='font-family: Arial, sans-serif; line-height: 1.5;'>
                <h4 style='color: #2ca02c; margin-top: 0;'>Response:</h4>
                <p style='color: #ffffff;'>{response_text}</p>
            </div>
            """
            display(widgets.HTML(value=html_content))

# Define what happens when you click clear
def on_clear_clicked(b):
    query_input.value = ''
    with response_output:
        clear_output()

# 3. Link the buttons to their respective functions
submit_button.on_click(on_submit_clicked)
clear_button.on_click(on_clear_clicked)

# 4. Display everything together
action_buttons = widgets.HBox([submit_button, clear_button], layout=widgets.Layout(margin='0 0 0 10px'))
input_box = widgets.HBox([query_input, action_buttons])
display(widgets.VBox([input_box, response_output]))